ST554_Final Project   
Author: Joy Zhou  
Date: 4/21/2026

# Introduction   
This project is designed to assess the ability to use Apache Spark for processing streaming data and for training machine learning models in a scalable computing environment.


We will use a modified dataset from the [UCI machine learning repository](https://archive.ics.uci.edu/dataset/849/power+consumption+of+tetouan+city) on power consumption in Tetouan City, the project focuses on modeling the relationship between electricity consumption across different city zones and key influencing factors such as time of day, temperature, and humidity.   


By leveraging Spark's data processing and machine learning libraries, a predictive model will be developed using historical data and then applied to incoming data streams to perform real-time monitoring and prediction of power consumption. This project integrates concepts from big data processing, machine learning, and streaming analytics, highlighting the practical application of Spark in handling real-world, data-intensive problems.


# Part 1 Fitting the Model



The dataset power_ml_data.csv is available at https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv.
The data should first be loaded into a pandas DataFrame using the `pd.read_csv()` function and then converted into a Spark DataFrame. In this analysis, `Power_Zone_3` is treated as the response variable, while all remaining variables are used as predictors.

## Read in data

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/27 10:27:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/27 10:27:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


- First we read `power_ml_data` into a standard pandas data frame named `power_data` using the pd.read_csv() function.


In [2]:
power_data = pd.read_csv("https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv")
power_data.head()

,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


- Convert `power_data` to a spark data frame `power`

In [3]:
#Convert this to a spark data frame
power = spark.createDataFrame(power_data)
power.show(5)
#We are going to treat the Power_Zone_3 variable as our response variable

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## Data transformation
We will fit an elastic net regression using cross-validation (CV), without performing an explicit training-test split. Instead, model selection and performance assessment are conducted entirely through cross-validation on the available dataset.   

Feature transformations are performed using Spark MLlib utilities, and the transformed variables are assembled into a machine learning pipeline to ensure a consistent and reproducible workflow.

- let's check the varaible types by inspectiong the dataset schema.


In [4]:
power.printSchema()

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



- Inspection of the schema shows that the Hour column is defined as a LongType. We therefore apply an SQLTransformer to cast this variable to DoubleType.

In [5]:
from pyspark.ml.feature import SQLTransformer

sqlTrans = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_double FROM __THIS__")

To inspect the results of the SQL transformation, we apply sqlTrans to the power dataset and displayed a sample of the transformed records.

In [6]:
#inspect of records of sqlTrans
transformed_power = sqlTrans.transform(power)
transformed_power.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335

- We apply a binarization step to the Hour variable using a cutoff of 6.5, creating a new binary feature, night_vs_day, to indicate nighttime (Hour < 6.5) versus daytime (Hour ≥ 6.5) for downstream modeling.
    

In [7]:
from pyspark.ml.feature import Binarizer

binaryTrans = Binarizer(threshold=6.5, inputCol="Hour_double", outputCol="night_vs_day")

#inspect the transformer results
binary = binaryTrans.transform(
    sqlTrans.transform(power))
binary.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|        0.0

- Although the Month variable is stored as a LongType, it represents a categorical feature rather than a continuous numeric value. Therefore, we apply one-hot encoding to the `Month` column to generate binary indicator variables, allowing the model to capture seasonal effects without assuming an ordinal or linear relationship among months.

In [8]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler

# Location one-hot encoding
month_encoder = OneHotEncoder(inputCol="Month", outputCol="Month_ind", dropLast=True)

We fit the month_encoder model to inspect the transformed records to ensure that the one-hot encoding is performed correctly.

In [10]:
#inspect the one-hot encode results
model = month_encoder.fit(binary)
encoded = model.transform(binary)
encoded.show(5)

+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|Hour_double|night_vs_day|     Month_ind|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+-----------+------------+--------------+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|        0.0|         0.0|(12,[1],[1.0])|
|      6.121|    75.0|     0.083|       


- Principal Component Analysis (PCA) is used to reduce the dimensionality of the environmental variables Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows. The variables are assembled into a feature vector and standardized using [StandardScaler](https://www.sparkcodehub.com/pyspark/mllib/standard-scaler) to mitigate the influence of variables with larger magnitudes. PCA is then applied to extract two uncorrelated principal components, which provide lower‑dimensional representations for subsequent analysis and modeling.


In [11]:
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline

#assemble raw features
assembler = VectorAssembler(inputCols=["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"], outputCol="pcaFeatures")

# Scale features (mean=0, std=1)
scaler = StandardScaler(inputCol="pcaFeatures", outputCol="scaledFeatures", withMean=True, withStd=True)

#PCA on scaled features
pca = PCA( k=2, inputCol="scaledFeatures", outputCol="pcaOutput")

#inspect the results
#set up pipeline
pipeline = Pipeline(stages=[assembler, scaler, pca])

# Fit and transform
pca_model = pipeline.fit(encoded)
pca_transformed = pca_model.transform(encoded)

pca_transformed.select("scaledFeatures", "pcaOutput").show(5, truncate=False)

+----------------------------------------------------------------------------------------------------+---------------------------------------+
|scaledFeatures                                                                                      |pcaOutput                              |
+----------------------------------------------------------------------------------------------------+---------------------------------------+
|[-2.1079477438809433,0.3542085264292604,-0.7996341497278044,-0.6900839531060153,-0.6025312519793238]|[2.0690743213463723,0.8078678829472259]|
|[-2.1328903699941857,0.3991947174962608,-0.7996341497278044,-0.6900121009460847,-0.6028048802947571]|[2.102921063880654,0.8152476222806391] |
|[-2.1502641992178924,0.3991947174962608,-0.800911098051248,-0.690042354487108,-0.6026841619203012]  |[2.1120662633791016,0.8227151924829956]|
|[-2.183291676554047,0.4313277111155467,-0.7996341497278044,-0.6899326854008982,-0.6027163534868227] |[2.1435031847422197,0.8329135817940964]|

As the output above, we combined several environmental measurements into a single dataset and used PCA to compress them into two summary variables that capture most of the information. These summaries were then used instead of the original variables in later analyses.

- we will rename our response variable `Power_Zone_3` as label

In [12]:
sqlLabel = SQLTransformer(
    statement = """
                SELECT *, Power_Zone_3 AS label FROM __THIS__
                """
)

- Use `VectorAssembler()` to put all predictors into a single features vector, which includes:   
    - two fitted PCA features (pcaOutput)      
    - a binary Hour variable (night_vs_day)   
    - Power_Zone_1   
    - Power_Zone_2   
    - Month indicator variables (Month_ind)   
Since the PCA features were standardized in previous steps, the additional non‑PCA features will also be standardized to ensure consistency across all predictors.

In [13]:
#assemble assembleFeatures
features_assembler = VectorAssembler(
    inputCols=["pcaOutput", "night_vs_day", "Power_Zone_1", "Power_Zone_2", "Month_ind"],
    outputCol="assembledFeatures"
)

#scale again after adding non‑PCA features.
final_scaler = StandardScaler(
    inputCol="assembledFeatures",
    outputCol="features",
    withMean=True,
    withStd=True
)

assembled_df = features_assembler.transform(pca_transformed)
scaler_model = final_scaler.fit(assembled_df)
featurestrans = scaler_model.transform(assembled_df)

# Inspect assembled features
featurestrans.select("assembledFeatures", "features").show(5, truncate=False)

+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|assembledFeatures                                                                   |features                                                                                                                                                                                                                                                                                                                              |
+------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------

From the output above, we confirmed that the transformation pipeline was completed and the the dataset is ready for modeling.

## Fit an Elastic Net Model
We next fit an Elastic Net regression model using the `CrossValidator()` and `LinearRegression()` function. Model hyperparameters are selected via cross-validation over a predefined grid consisting of all combinations of the following values:   

- regParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1
- elasticNetParam: 0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1

We are now ready to set up the pipeline, which includes transcormations and the model to be fitted. We use the `Pipeline()` function from the `pyspark.ml` module to set up a squential workflow of transformations/estimators.

In [14]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml import Pipeline

# Elastic Net regression
lr = LinearRegression(featuresCol='features', labelCol='label') #create a liner regression instance

#define parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .addGrid(lr.elasticNetParam, [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]) \
    .build()

- Set up the pipeline 

In [15]:
pipeline = Pipeline(stages=[
    sqlTrans,              # Cast hour 
    binaryTrans,           # night_vs_day
    month_encoder,         # Month one-hot encoding
    sqlLabel,              # Rename response variable
    assembler,             # Assemble environmental vars to create pcaFeatures
    scaler,                # Scale pcaFeatures
    pca,                   # PCA produces pcaOutput
    features_assembler,    # Combine pcaOutput + indicators to get assembledFeatures
    final_scaler,          # Scale final feature vector to features
    lr                     # Elastic Net regression model
])

We apply `CrossValidator()` to carry out 5-fold cross-validation over the specified hyperparameter grid. Model performance is evaluated using `RegressionEvaluator()`, with RMSE specified via the metricName argument. This procedure will split the dataset into five folds. In each iteration, the model is trained on four folds and validated on the remaining fold for every hyperparameter combination. The performance metric (rmse) is then averaged across all five folds, and the model with the lowest average rmse is selected as the best model.

In [17]:
from pyspark.ml.evaluation import RegressionEvaluator
#initialize cross-validator
cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    ),
    numFolds=5,    #5-fold cross-validation
    parallelism=128,
    seed = 42
)
#fitting the model, and choose the best set of parameters
cv_model = cv.fit(power)

26/04/27 10:29:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/27 10:29:48 WARN Instrumentation: [15879d17] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 10:29:53 WARN Instrumentation: [15879d17] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/27 10:29:57 WARN Instrumentation: [43cf8851] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 10:29:57 WARN Instrumentation: [762372b3] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 10:29:59 WARN Instrumentation: [9181c12d] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 10:29:59 WARN Instrumentation: [457e1dd1] regParam is zero, which might cause numerical instability and overfitting.
26/04/27 10:29:59 WARN Instrumentatio

Evaluating the best model
- We will use the bestModel attribute to retrive the best model from cross validation
- Then, extract the final stage of the pipeline, the trained LinearRegression model, from the selected best model for further evaluation and interpretation.

In [18]:
# Retrieves the best pipeline model from the cross-validation
best_model = cv_model.bestModel
#Extracts the last stage of the pipeline
best_lr = best_model.stages[-1]  # LinearRegression is last stage

Report the optimal values of the tuning parameters (regParam and elasticNetParam) selected by cross‑validation using `getRegParam()` and `getElasticNetParam()` methods.

In [19]:
# Printing parameters for verification
print(f"Best regParam: {best_lr.getRegParam()}")
print(f"Best elasticNetParam: {best_lr.getElasticNetParam()}")

Best regParam: 0.05
Best elasticNetParam: 0.05


Both parameters were selected as 0.05 because cross‑validation identified a model with mild, Ridge‑dominant regularization that best balances bias and variance for the standardized feature set.

Report the CV error   
- The cross‑validation error is computed as the average RMSE across the five folds, evaluated over all hyperparameter combinations.

In [20]:
#report CV error
avg_rmse = min(cv_model.avgMetrics)
print(f"Best CV RMSE: {avg_rmse:.4f}")

Best CV RMSE: 2174.9962


Report training set RMSE
- The fitted regression model with the best parameters now has a transform method that can be used to make predictions using the entire dataset. The training set RMSE is obtained by applying the best fitted pipeline model to the entire dataset using the transform() method and evaluating prediction error using RMSE.

In [21]:
#apply the best fitted pipeline model to the entire dataset
train_predictions = best_model.transform(power)

#define evaluator
evaluator=RegressionEvaluator(
        labelCol='label',
        predictionCol='prediction',
        metricName='rmse'
    )
# Evaluate RMSE
train_rmse = evaluator.evaluate(train_predictions)
print(f"Training RMSE: {train_rmse:.4f}")

Training RMSE: 2174.4490


The model outputs are used to generate predictions, from which a residual column is computed as the difference between the observed label and the predicted value. The `.withColumn()` method is used to create this residual. A resulting data frame is then displayed containing the residuals, the label column, and the model predictions.

In [22]:
from pyspark.sql.functions import col

residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction"))

residuals_df.select( "label", "prediction", "residual",).show(10, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20936.226841006068|-695.262981006068 |
|20131.08434|18701.58646809512 |1429.4978719048813|
|19668.43373|18237.189469773388|1431.2442602266128|
|18899.27711|17615.892216250577|1283.3848937494222|
|18442.40964|17017.38184268389 |1425.02779731611  |
|18130.12048|16540.614302205624|1589.5061777943774|
|17945.06024|16114.684233894426|1830.3760061055727|
|17459.27711|15742.03833420217 |1717.2387757978286|
|17025.54217|15282.557973009165|1742.9841969908357|
|16794.21687|14934.899321422641|1859.317548577359 |
+-----------+------------------+------------------+
only showing top 10 rows


## Conclusion
Using the Power dataset, an elastic net linear regression model was fitted through a pipeline and tuned via 5‑fold cross‑validation, with RMSE serving as the evaluation metric. The optimal regularization parameters were selected based on cross‑validated performance. The best‑fitted pipeline model was then applied to the entire dataset to compute the training set RMSE, which was 2174.449. Prediction residuals were subsequently calculated as the difference between the observed and predicted values.
During this process, transformations were applied to the hour and month variables, and all features were standardized prior to PCA and model fitting to mitigate the influence of variables with larger magnitudes. The use of a pipeline enabled efficient tracking of feature transformations and streamlined the modeling workflow, while Spark facilitated scalable and efficient model fitting. In the next section, methods for handling streaming data will be explored.

# Streaming Part


## Reading a stream
We set up a Structured Streaming read operation that monitors a specified folder for incoming CSV files, which created by .py file).

In [23]:
#create spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

- The schema from the power DataFrame is extracted and reused to ensure consistent column types when setting up the streaming CSV source. The header option is set to True so that all incoming files are correctly read with column headers.

In [24]:
#set up schema
power_schema = power.schema

#set up the stream df
stream_df = spark.readStream.schema(power_schema).option("header", True).csv("csv_files")

## Transform/Aggregation Step
The streaming dataset is processed using two separate transformations derived from the same input stream, following these steps:

- The first transformation applies the fitted pipeline model to generate prediction values, residuals are computed as the difference between the observed values and the model predictions.

In [25]:
from pyspark.sql.functions import col

# Apply the fitted feature-engineering pipeline and the model to streaming data
predictions_df = (
    best_model
        .transform(stream_df)
        .withColumn("residual", col("label") - col("prediction"))     #Create a residual column
        .select("label", "prediction", "residual")
)

- The second transformation operates on the original stream and modifies the response variable `Power_Zone_3` to be named label.

In [26]:
#2nd treansformation
label_df = stream_df.withColumn("label", col("Power_Zone_3")).drop("Power_Zone_3")

- The two transformed streams are then joined on the common label column using `join()` method.

In [27]:
joined_stream = predictions_df.join(label_df, on="label")

## Write the stream

The `.writeStream` method is used to output the transformed streaming data to the console in append mode. The `.start()` method initiates the streaming query and begins monitoring the input directory for new data. The five CSV files are added to the empty `csv_files` folder one at a time to be processed incrementally.

In [28]:
#Write the transformed streaming data to the console and start the streaming query
query = (
    joined_stream
        .writeStream
        .format("console") 
        .outputMode("append")
        .start()
)

26/04/27 11:19:25 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-bd6bfe93-b301-4359-82f8-19fa55ac0e76. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 11:19:25 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14036.67692|14859.103605327562|-822.4266853275622|      21.42|    80.7|     0.069|                563.7|        531.8| 29537.48344| 19313.51351|    6|   9|
|25080.37618|27205.333971150052|-2124.957791150053|      29.89|   55.25|     4.905|                195.9|        148.7| 40556.53718| 32114.88912|    8|  16|
|19299.20973|20518.371041804407|-1219.161311804408|       19.9|    78.5|     0.069|                0.051|        0.104

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10796.35258| 8091.147202688515|  2705.205377311486|      17.68|   69.21|     0.085|                0.059|         0.13|  22895.0547|  17447.3029|   10|   2|
|    15936.0|16120.026874435083| -184.0268744350833|      13.34|    76.8|     0.085|                0.026|        0.245| 24813.26157| 13813.44196|    4|   1|
|10690.41879|12401.609320779411|-1711.1905307794113|      21.04|    93.7|     4.917|                20.43|       

-------------------------------------------
Batch: 2
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|6223.289316|2993.1337958953955|  3230.155520104605|       9.03|    85.4|     0.083|                0.033|        0.156| 18263.11787| 13185.63977|   12|   7|
|15224.51613| 12814.12411889244| 2410.3920111075604|      11.93|    82.0|     0.082|                 0.04|        0.141| 22316.93617| 12962.19512|    3|   4|
|8572.887538| 9881.043826142777|-1308.1562881427762|      19.53|   58.53|     0.098|                 84.1|       

-------------------------------------------
Batch: 3
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|10961.45455|12191.000667071134|-1229.5461170711333|      13.08|    83.1|     0.082|                0.022|        0.234| 18960.25834| 10316.08961|    4|   5|
|24154.64435|24454.299335116317|-299.65498511631813|       26.6|   59.11|     4.919|                821.0|        127.8| 32754.81728| 19435.44304|    7|  12|
|29827.93846| 29921.49224952139| -93.55378952138926|      24.67|   43.96|     4.917|                10.26|       

...

-------------------------------------------
Batch: 15
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17615.75879| 17584.06562539178|31.693164608219377|      16.63|   37.72|     0.086|                651.8|        57.47| 33516.61017| 20312.46201|    2|  13|
|12590.80695|13149.163917565988|-558.3569675659874|       22.6|    74.0|     4.918|                592.5|         84.3| 31603.53982| 21805.82121|    9|  10|
|14333.42714|13348.919921486111| 984.5072185138888|       8.34|    90.6|     0.083|                 0.04|        0.15

-------------------------------------------
Batch: 16
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|24585.64824|23722.842292047175| 862.8059479528238|      13.94|   63.93|     0.075|                0.055|        0.111| 40777.62712| 24995.74468|    2|  21|
|41314.14226|34165.692613768784| 7148.449646231216|      25.66|    77.2|     4.913|                0.234|        0.207| 44077.07641| 30653.16456|    7|  20|
|9657.623049| 6754.856599356397|2902.7664496436028|      11.66|    53.3|     0.078|                0.095|        0.08

-------------------------------------------
Batch: 17
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|20972.90323|19407.282277770497|  1565.620952229503|      17.04|    87.2|     0.066|                0.022|        0.108| 34890.89362| 20341.46341|    3|  22|
|10466.84372|12142.557921954878|-1675.7142019548774|      18.91|    84.8|     4.916|                177.4|         94.8| 29653.80531| 18924.32432|    9|   8|
|31195.48589|26155.192197938708|  5040.293692061292|      23.29|   60.12|     4.901|                 0.08|      

-------------------------------------------
Batch: 18
-------------------------------------------
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|          residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|17960.72727| 16300.26521486783|1660.4620551321696|      26.33|   35.74|     0.071|                188.5|        201.0| 29699.03122| 17043.17719|    4|  18|
|31583.59833|27988.470789043917| 3595.127540956084|      32.38|   50.55|     4.909|                649.3|         81.0| 36977.54153| 25120.25316|    7|  16|
|28076.98745| 30450.50901253887| -2373.52156253887|      34.17|   29.74|     4.911|                829.0|        37.9

-------------------------------------------
Batch: 19
-------------------------------------------
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|      label|        prediction|           residual|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Month|Hour|
+-----------+------------------+-------------------+-----------+--------+----------+---------------------+-------------+------------+------------+-----+----+
|14185.16129|15346.686304887993|-1161.5250148879932|      14.68|   69.39|     0.076|                385.8|        365.5| 31251.06383| 19745.12195|    3|  10|
|29696.80251|26969.684950707677| 2727.1175592923246|       28.6|   31.81|     4.908|                0.084|        0.144| 37040.44395| 27074.12883|    8|   0|
|18810.18182| 18404.46924914815|  405.7125708518506|      24.71|   39.91|     4.918|                352.5|      

When We start the query, We read in streaming datasets using the method that created in `stream.py` file. We will submit `stream.py` in a python console.


In [29]:
# Stop the streaming query after processing all input files
query.stop()

26/04/27 11:27:35 WARN DAGScheduler: Failed to cancel job group 20ba20c2-5894-4b38-afbe-6bcd8e489d4f. Cannot find active jobs for it.
26/04/27 11:27:35 WARN DAGScheduler: Failed to cancel job group 20ba20c2-5894-4b38-afbe-6bcd8e489d4f. Cannot find active jobs for it.
